# Tema 2 - Analisis Cinematico de Mecanismos Planos

**Teoria de Maquinas y Mecanismos** | Universidad

---

## Objetivos de aprendizaje

1. Plantear y resolver el **problema de posicion** (no lineal) usando `fsolve`
2. Plantear y resolver el **problema de velocidad** (lineal) usando el Jacobiano
3. Plantear y resolver el **problema de aceleracion** (lineal) usando el Jacobiano
4. Aplicar los tres metodos: velocidades relativas, ecuaciones de lazo y CIR
5. Analizar mecanismos de 4 barras, biela-manivela y multi-lazo

---

## Indice

1. [Formulario del Tema](#1.-Formulario-del-Tema)
2. [Introduccion](#2.-Introduccion)
3. [Metodo de velocidades y aceleraciones relativas](#3.-Metodo-de-velocidades-y-aceleraciones-relativas)
4. [Metodo de ecuaciones de lazo](#4.-Metodo-de-ecuaciones-de-lazo)
5. [Metodo de centros instantaneos de rotacion](#5.-Metodo-de-centros-instantaneos-de-rotacion-(CIR))
6. [Ejemplo completo: 4-barras](#6.-Ejemplo-completo:-4-barras)
7. [Ejemplo completo: biela-manivela-corredera](#7.-Ejemplo-completo:-biela-manivela-corredera)
8. [Ejemplo: mecanismo multi-lazo (Stephenson III)](#8.-Ejemplo:-mecanismo-multi-lazo)
9. [Ejercicios resueltos](#9.-Ejercicios-resueltos)
10. [Catalogo completo de ejercicios](#10.-Catalogo-completo-de-ejercicios)
11. [Resumen y formulas clave](#11.-Resumen-y-formulas-clave)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Arc, Circle, Rectangle, Polygon
from matplotlib.lines import Line2D
from scipy.optimize import fsolve

# --- Paleta de colores estandar ---
COLOR_PRINCIPAL = '#2171b5'   # Azul - barras, curvas principales
COLOR_FIJO      = '#636363'   # Gris oscuro - barra fija / suelo
COLOR_PUNTO     = '#238b45'   # Verde - puntos de operacion, resultados
COLOR_ROJO      = '#cb181d'   # Rojo - fuerzas, errores, alertas
COLOR_NARANJA   = '#ff7f00'   # Naranja - secundario
COLOR_MORADO    = '#6a3d9a'   # Morado - terciario
COLOR_CLARO     = '#a6cee3'   # Azul claro - rellenos

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.grid': True,
    'axes.grid.which': 'major',
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'figure.dpi': 100,
    'figure.figsize': (10, 5),
})

# --- Helpers de dibujo para mecanismos ---
def draw_link(ax, p1, p2, color=COLOR_PRINCIPAL, lw=5, zorder=2):
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], '-', color=color, lw=lw,
            solid_capstyle='round', zorder=zorder)

def draw_joint(ax, p, fixed=False, color='black', ms=10, zorder=5):
    if fixed:
        ax.plot(p[0], p[1], 's', color=color, ms=ms, zorder=zorder,
                markeredgecolor='black', markeredgewidth=1.5)
    else:
        ax.plot(p[0], p[1], 'o', color='white', ms=ms, zorder=zorder,
                markeredgecolor='black', markeredgewidth=2)

def draw_ground(ax, center, width, y_offset=-0.15):
    x0 = center[0] - width/2
    y0 = center[1] + y_offset
    n_hash = 8
    for i in range(n_hash):
        xi = x0 + i * width / (n_hash - 1)
        ax.plot([xi, xi - 0.08], [y0, y0 - 0.12], 'k-', lw=0.8, zorder=1)
    ax.plot([x0, x0 + width], [y0, y0], 'k-', lw=1.5, zorder=1)

def draw_slider(ax, p, angle=0, size=0.3, color=COLOR_FIJO):
    rect = Rectangle((p[0]-size/2, p[1]-size/4), size, size/2,
                      angle=angle, facecolor=color, edgecolor='black',
                      lw=1.5, alpha=0.3, zorder=3)
    ax.add_patch(rect)

def draw_velocity_arrow(ax, origin, vec, scale=1.0, color=COLOR_ROJO, label=None):
    """Draw a velocity vector arrow from origin."""
    ax.annotate('', xy=(origin[0]+vec[0]*scale, origin[1]+vec[1]*scale),
                xytext=origin,
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    if label:
        mid = (origin[0]+vec[0]*scale/2, origin[1]+vec[1]*scale/2)
        ax.text(mid[0]+0.05, mid[1]+0.05, label, fontsize=9, color=color,
                fontweight='bold')

## 1. Formulario del Tema

| Concepto | Formula | Notas |
|:---|:---:|:---|
| **Ecuacion de cierre de lazo (posicion)** | $\boxed{\sum_{i} L_i \begin{bmatrix} \cos\theta_i \\ \sin\theta_i \end{bmatrix} = \mathbf{0}}$ | No lineal en las incognitas $\theta_i$ |
| **Posicion (fsolve)** | $\boxed{\mathbf{f}(\boldsymbol{\theta}) = \mathbf{0} \;\Rightarrow\; \text{fsolve}(\mathbf{f}, \boldsymbol{\theta}_0)}$ | Necesita semilla $\boldsymbol{\theta}_0$ (warm-start) |
| **Jacobiano de velocidad** | $\boxed{[\mathbf{J}]\{\dot{\boldsymbol{\theta}}\} = \{\mathbf{b}_{vel}\}}$ | $J_{ij} = \partial f_i / \partial \theta_j$. Sistema **lineal** |
| **Jacobiano de aceleracion** | $\boxed{[\mathbf{J}]\{\ddot{\boldsymbol{\theta}}\} = \{\mathbf{b}_{acel}\}}$ | Mismo $\mathbf{J}$. $\mathbf{b}_{acel}$ incluye terminos $\omega^2$ |
| **Velocidad relativa** | $\boxed{\mathbf{v}_P = \mathbf{v}_O + \boldsymbol{\omega} \times \overrightarrow{OP}}$ | $\boldsymbol{\omega} = \omega\,\hat{k}$ para mecanismos planos |
| **Aceleracion relativa** | $\boxed{\mathbf{a}_P = \mathbf{a}_O + \boldsymbol{\alpha} \times \overrightarrow{OP} - \omega^2 \overrightarrow{OP}}$ | Termino centripeto $-\omega^2 \overrightarrow{OP}$ |
| **CIR (centro instantaneo)** | $\boxed{\mathbf{v}_P = \boldsymbol{\omega} \times (\mathbf{P} - \mathbf{CIR})}$ | $\|\mathbf{v}_P\| = \|\omega\| \cdot d(P, CIR)$ |
| **Par de rotacion en A (vel.)** | $\boxed{\mathbf{v}_{21}^A = \mathbf{v}_{31}^A}$ | Misma velocidad en la junta |
| **Par de rotacion en A (acel.)** | $\boxed{\mathbf{a}_{21}^A = \mathbf{a}_{31}^A}$ | Misma aceleracion en la junta |
| **Teorema de Kennedy** | $\boxed{I_{ij},\; I_{jk},\; I_{ik} \text{ son colineales}}$ | Para encontrar CIR de 3 cuerpos |


## 2. Introduccion

El analisis cinematico de un mecanismo plano consiste en resolver **tres problemas secuenciales**:

| Problema | Naturaleza | Metodo de resolucion |
|:---|:---:|:---|
| **1. Posicion** | **No lineal** | `fsolve` (Newton-Raphson) |
| **2. Velocidad** | **Lineal** | $[\mathbf{J}]\{\dot{\theta}\} = \{\mathbf{b}_{vel}\}$ |
| **3. Aceleracion** | **Lineal** | $[\mathbf{J}]\{\ddot{\theta}\} = \{\mathbf{b}_{acel}\}$ |

**Dato clave**: una vez resuelto el problema de posicion (el mas dificil por ser no lineal),
los problemas de velocidad y aceleracion comparten la **misma matriz Jacobiana**.
Solo cambia el vector del lado derecho.

### Tres metodos de analisis

1. **Velocidades y aceleraciones relativas**: aplica directamente las ecuaciones vectoriales de la cinematica del solido rigido.
2. **Ecuaciones de lazo**: plantea las restricciones geometricas de cierre y las deriva respecto al tiempo.
3. **Centros instantaneos de rotacion (CIR)**: metodo grafico basado en el teorema de Kennedy.

Los tres metodos son equivalentes. El metodo de ecuaciones de lazo es el mas sistematico
y el que mejor se presta a la implementacion computacional.


## 3. Metodo de velocidades y aceleraciones relativas

### 3.1 Resumen de ecuaciones

Para un cuerpo rigido $i$ que rota respecto al sistema fijo (cuerpo 1),
la **velocidad** de un punto $P$ del cuerpo $i$ es:

$$\mathbf{v}_{P,i1} = \mathbf{v}_{O,i1} + \omega_{i1}\,\hat{k} \times \overrightarrow{OP}$$

En componentes (mecanismo plano con $\omega = \omega\,\hat{k}$):

$$\begin{cases} v_{Px} = v_{Ox} - \omega \cdot (P_y - O_y) \\ v_{Py} = v_{Oy} + \omega \cdot (P_x - O_x) \end{cases}$$

La **aceleracion** de $P$:

$$\mathbf{a}_{P,i1} = \mathbf{a}_{O,i1} + \alpha_{i1}\,\hat{k} \times \overrightarrow{OP} - \omega_{i1}^2 \cdot \overrightarrow{OP}$$

En componentes:

$$\begin{cases} a_{Px} = a_{Ox} - \alpha(P_y - O_y) - \omega^2(P_x - O_x) \\ a_{Py} = a_{Oy} + \alpha(P_x - O_x) - \omega^2(P_y - O_y) \end{cases}$$

### 3.2 Aplicacion a pares de rotacion

Si los cuerpos $i$ y $j$ estan unidos por un **par de rotacion** en el punto $A$:

$$\boxed{\mathbf{v}_{i1}^A = \mathbf{v}_{j1}^A} \qquad \boxed{\mathbf{a}_{i1}^A = \mathbf{a}_{j1}^A}$$

Esto genera **2 ecuaciones escalares** (componentes $x$ e $y$) por cada par de rotacion.

### 3.3 Aplicacion a pares prismaticos

Si los cuerpos $i$ y $j$ estan unidos por un **par prismatico** (corredera):

- La velocidad relativa es **tangente a la guia**: $\mathbf{v}_{j/i} = \dot{d}\,\hat{e}_t$
- La aceleracion incluye **termino de Coriolis**: $\mathbf{a}_{Coriolis} = 2\omega_i \times \mathbf{v}_{j/i}$

$$\mathbf{a}_{j1} = \mathbf{a}_{i1} + \boldsymbol{\alpha}_i \times \overrightarrow{r} - \omega_i^2\overrightarrow{r} + \ddot{d}\,\hat{e}_t + 2\omega_i \times \dot{d}\,\hat{e}_t$$


In [ ]:
# --- Diagrama de mecanismo de 4 barras con vectores de velocidad ---
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.set_aspect('equal')
ax.set_xlim(-0.5, 6.5)
ax.set_ylim(-1.0, 5.5)
ax.set_title('Mecanismo de 4 barras: velocidades relativas', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.2)

# Posiciones de ejemplo
O2 = np.array([0.0, 0.0])
O4 = np.array([5.0, 0.0])
th2 = np.radians(60)
L2, L3, L4 = 1.5, 3.5, 3.0
A = O2 + L2 * np.array([np.cos(th2), np.sin(th2)])
# Solve for B approximately
th4 = np.radians(75)
B = O4 + L4 * np.array([np.cos(th4), np.sin(th4)])

# Draw links
draw_link(ax, O2, O4, color=COLOR_FIJO, lw=8)      # Barra fija (1)
draw_link(ax, O2, A, color=COLOR_ROJO, lw=5)        # Manivela (2)
draw_link(ax, A, B, color=COLOR_PRINCIPAL, lw=5)     # Biela (3)
draw_link(ax, O4, B, color=COLOR_PUNTO, lw=5)        # Balancin (4)

# Draw joints
draw_joint(ax, O2, fixed=True)
draw_joint(ax, O4, fixed=True)
draw_joint(ax, A)
draw_joint(ax, B)
draw_ground(ax, O2, 0.6)
draw_ground(ax, O4, 0.6)

# Labels
ax.text(2.5, -0.5, '1 (fijo)', fontsize=11, ha='center', color=COLOR_FIJO, fontweight='bold')
ax.text(O2[0]-0.15, A[1]/2, '2', fontsize=12, ha='right', color=COLOR_ROJO, fontweight='bold')
ax.text((A[0]+B[0])/2, (A[1]+B[1])/2+0.3, '3', fontsize=12, ha='center', color=COLOR_PRINCIPAL, fontweight='bold')
ax.text(O4[0]+0.3, B[1]/2, '4', fontsize=12, ha='left', color=COLOR_PUNTO, fontweight='bold')
ax.text(O2[0]-0.2, O2[1]-0.4, '$O_2$', fontsize=12, ha='center')
ax.text(O4[0]+0.2, O4[1]-0.4, '$O_4$', fontsize=12, ha='center')
ax.text(A[0]-0.3, A[1]+0.2, '$A$', fontsize=12, ha='center')
ax.text(B[0]+0.2, B[1]+0.2, '$B$', fontsize=12, ha='center')

# Velocity vectors (illustrative)
omega2 = -20  # rad/s (into page)
# v_A = omega2 x O2A  => v_A = omega2 * L2 * [-sin(th2), cos(th2)]
vA = omega2 * L2 * np.array([-np.sin(th2), np.cos(th2)])
vA_norm = vA / np.linalg.norm(vA)
draw_velocity_arrow(ax, A, vA_norm, scale=1.2, color=COLOR_ROJO, label=r'$\mathbf{v}_A$')

# v_B direction (perpendicular to O4B)
vB_dir = np.array([-np.sin(th4), np.cos(th4)])
draw_velocity_arrow(ax, B, vB_dir, scale=1.2, color=COLOR_PUNTO, label=r'$\mathbf{v}_B$')

# omega arrows (curved)
arc2 = Arc(O2, 0.8, 0.8, angle=0, theta1=20, theta2=np.degrees(th2)-5, color=COLOR_ROJO, lw=1.5)
ax.add_patch(arc2)
ax.annotate('', xy=(O2[0]+0.35, O2[1]+0.25), xytext=(O2[0]+0.4, O2[1]+0.15),
            arrowprops=dict(arrowstyle='->', color=COLOR_ROJO, lw=1.5))
ax.text(O2[0]+0.5, O2[1]+0.35, r'$\omega_2$', fontsize=11, color=COLOR_ROJO)

# Equation box
eq_text = (r'$\mathbf{v}_A = \omega_2 \times \overrightarrow{O_2 A}$' + '\n'
           r'$\mathbf{v}_B = \mathbf{v}_A + \omega_3 \times \overrightarrow{AB}$' + '\n'
           r'$\mathbf{v}_B = \omega_4 \times \overrightarrow{O_4 B}$')
ax.text(0.5, 4.5, eq_text, fontsize=11, va='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor=COLOR_CLARO, alpha=0.5))

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
plt.tight_layout()
plt.show()

## 4. Metodo de ecuaciones de lazo

Este es el metodo mas sistematico y el que mejor se adapta a la programacion.
Se basa en escribir las **ecuaciones de cierre geometrico** del mecanismo
y derivarlas respecto al tiempo.

### 4.1 Ecuaciones de cierre (posicion)

Para un mecanismo de 4 barras con un solo lazo:

$$\overrightarrow{O_2 A} + \overrightarrow{AB} = \overrightarrow{O_2 O_4} + \overrightarrow{O_4 B}$$

En componentes:

$$\boxed{\begin{cases} L_2 \cos\theta_2 + L_3 \cos\theta_3 - L_1 - L_4 \cos\theta_4 = 0 \\ L_2 \sin\theta_2 + L_3 \sin\theta_3 - L_4 \sin\theta_4 = 0 \end{cases}}$$

Dado $\theta_2$ (entrada), se resuelve el sistema **no lineal** para $(\theta_3, \theta_4)$ usando `fsolve`.

### 4.2 Primera derivada $\Rightarrow$ velocidades (Jacobiano)

Derivando las ecuaciones de cierre respecto al tiempo:

$$\boxed{\underbrace{\begin{bmatrix} -L_3 \sin\theta_3 & L_4 \sin\theta_4 \\ L_3 \cos\theta_3 & -L_4 \cos\theta_4 \end{bmatrix}}_{\mathbf{J}} \begin{Bmatrix} \dot\theta_3 \\ \dot\theta_4 \end{Bmatrix} = \underbrace{\begin{Bmatrix} L_2 \dot\theta_2 \sin\theta_2 \\ -L_2 \dot\theta_2 \cos\theta_2 \end{Bmatrix}}_{\mathbf{b}_{vel}}}$$

### 4.3 Segunda derivada $\Rightarrow$ aceleraciones

Derivando de nuevo:

$$\boxed{\mathbf{J} \begin{Bmatrix} \ddot\theta_3 \\ \ddot\theta_4 \end{Bmatrix} = \underbrace{\begin{Bmatrix} L_2 \ddot\theta_2 \sin\theta_2 + L_2 \dot\theta_2^2 \cos\theta_2 + L_3 \dot\theta_3^2 \cos\theta_3 - L_4 \dot\theta_4^2 \cos\theta_4 \\ -L_2 \ddot\theta_2 \cos\theta_2 + L_2 \dot\theta_2^2 \sin\theta_2 + L_3 \dot\theta_3^2 \sin\theta_3 - L_4 \dot\theta_4^2 \sin\theta_4 \end{Bmatrix}}_{\mathbf{b}_{acel}}}$$

**Nota clave**: la matriz $\mathbf{J}$ es la **misma** para velocidades y aceleraciones.
Solo cambia el lado derecho.


In [ ]:
# =============================================================================
# ANALISIS CINEMATICO COMPLETO DE UN MECANISMO DE 4 BARRAS
# Metodo: ecuaciones de lazo + Jacobiano
# =============================================================================

# --- Parametros del mecanismo ---
L1 = 0.50   # Barra fija (distancia O2-O4) [m]
L2 = 0.15   # Manivela (entrada) [m]
L3 = 0.40   # Biela (acoplador) [m]
L4 = 0.35   # Balancin (salida) [m]
omega2 = 10.0    # Velocidad angular de entrada [rad/s] (constante)
alpha2 = 0.0     # Aceleracion angular de entrada [rad/s^2]

# --- Ecuaciones de cierre de lazo (posicion) ---
def loop_closure(x, theta2):
    """Ecuaciones de cierre para el 4-barras.
    x = [theta3, theta4]
    """
    theta3, theta4 = x
    f1 = L2*np.cos(theta2) + L3*np.cos(theta3) - L1 - L4*np.cos(theta4)
    f2 = L2*np.sin(theta2) + L3*np.sin(theta3) - L4*np.sin(theta4)
    return [f1, f2]

# --- Barrido completo de theta2 ---
N = 360
theta2_array = np.linspace(0, 2*np.pi, N, endpoint=False)

theta3_array = np.zeros(N)
theta4_array = np.zeros(N)
omega3_array = np.zeros(N)
omega4_array = np.zeros(N)
alpha3_array = np.zeros(N)
alpha4_array = np.zeros(N)

# Semilla inicial (warm-start)
x0 = [np.radians(30), np.radians(60)]  # Estimacion inicial para theta3, theta4

for i, theta2 in enumerate(theta2_array):
    # 1. POSICION: resolver sistema no lineal con fsolve
    sol = fsolve(loop_closure, x0, args=(theta2,), full_output=True)
    x_sol = sol[0]
    theta3, theta4 = x_sol
    theta3_array[i] = theta3
    theta4_array[i] = theta4
    
    # Warm-start: usar solucion actual como semilla para la siguiente iteracion
    x0 = [theta3, theta4]
    
    # 2. VELOCIDAD: construir Jacobiano y resolver sistema lineal
    J = np.array([
        [-L3*np.sin(theta3),  L4*np.sin(theta4)],
        [ L3*np.cos(theta3), -L4*np.cos(theta4)]
    ])
    
    b_vel = np.array([
        L2*omega2*np.sin(theta2),
        -L2*omega2*np.cos(theta2)
    ])
    
    omega_sol = np.linalg.solve(J, b_vel)
    omega3 = omega_sol[0]
    omega4 = omega_sol[1]
    omega3_array[i] = omega3
    omega4_array[i] = omega4
    
    # 3. ACELERACION: mismo Jacobiano, distinto lado derecho
    b_acel = np.array([
        L2*alpha2*np.sin(theta2) + L2*omega2**2*np.cos(theta2)
        + L3*omega3**2*np.cos(theta3) - L4*omega4**2*np.cos(theta4),
        -L2*alpha2*np.cos(theta2) + L2*omega2**2*np.sin(theta2)
        + L3*omega3**2*np.sin(theta3) - L4*omega4**2*np.sin(theta4)
    ])
    
    alpha_sol = np.linalg.solve(J, b_acel)
    alpha3_array[i] = alpha_sol[0]
    alpha4_array[i] = alpha_sol[1]

In [ ]:
# --- Graficas: posicion, velocidad, aceleracion ---
theta2_deg = np.degrees(theta2_array)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Posicion
axes[0].plot(theta2_deg, np.degrees(theta3_array), '-', color=COLOR_PRINCIPAL, lw=2, label=r'$\theta_3$')
axes[0].plot(theta2_deg, np.degrees(theta4_array), '-', color=COLOR_PUNTO, lw=2, label=r'$\theta_4$')
axes[0].set_ylabel('Angulo (grados)', fontsize=12)
axes[0].set_title('Analisis cinematico del mecanismo de 4 barras', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11, loc='upper right')

# Velocidad
axes[1].plot(theta2_deg, omega3_array, '-', color=COLOR_PRINCIPAL, lw=2, label=r'$\omega_3$')
axes[1].plot(theta2_deg, omega4_array, '-', color=COLOR_PUNTO, lw=2, label=r'$\omega_4$')
axes[1].set_ylabel('Vel. angular (rad/s)', fontsize=12)
axes[1].legend(fontsize=11, loc='upper right')

# Aceleracion
axes[2].plot(theta2_deg, alpha3_array, '-', color=COLOR_PRINCIPAL, lw=2, label=r'$\alpha_3$')
axes[2].plot(theta2_deg, alpha4_array, '-', color=COLOR_PUNTO, lw=2, label=r'$\alpha_4$')
axes[2].set_ylabel('Acel. angular (rad/s$^2$)', fontsize=12)
axes[2].set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
axes[2].legend(fontsize=11, loc='upper right')

for ax in axes:
    ax.axhline(0, color='gray', lw=0.5, ls='--')

plt.tight_layout()
plt.show()

## 5. Metodo de centros instantaneos de rotacion (CIR)

### Definicion

El **centro instantaneo de rotacion** $I_{ij}$ de dos cuerpos $i$ y $j$ es el punto
donde la velocidad relativa entre ambos cuerpos es cero:

$$\mathbf{v}_{I_{ij}, i} = \mathbf{v}_{I_{ij}, j}$$

Para un mecanismo plano, todo movimiento relativo entre dos cuerpos es una rotacion
instantanea alrededor de su CIR.

### Teorema de Kennedy

Dados tres cuerpos $i$, $j$, $k$ en movimiento relativo plano:

$$\boxed{I_{ij},\; I_{jk},\; I_{ik} \quad \text{son colineales}}$$

### Numero de CIR

Para un mecanismo de $n$ cuerpos, el numero de CIR es:

$$N_{CIR} = \frac{n(n-1)}{2}$$

Para un mecanismo de 4 barras ($n=4$): $N_{CIR} = 6$

### Aplicacion a velocidades

Una vez conocido el CIR $I_{i1}$ del cuerpo $i$ respecto al fijo:

$$\boxed{v_P = \omega_i \cdot d(P, I_{i1})}$$

donde $d(P, I_{i1})$ es la distancia del punto $P$ al CIR.
La velocidad es perpendicular a la linea $P - I_{i1}$.


In [ ]:
# --- Diagrama de centros instantaneos de rotacion para un 4-barras ---
fig, ax = plt.subplots(1, 1, figsize=(11, 8))
ax.set_aspect('equal')
ax.set_title('Centros instantaneos de rotacion (CIR) - Mecanismo de 4 barras',
             fontsize=13, fontweight='bold')

# Parametros del mecanismo
O2_cir = np.array([0.0, 0.0])
O4_cir = np.array([4.0, 0.0])
th2_cir = np.radians(55)
L2c, L3c, L4c = 1.5, 3.2, 2.8
A_cir = O2_cir + L2c * np.array([np.cos(th2_cir), np.sin(th2_cir)])
th4_cir = np.radians(110)
B_cir = O4_cir + L4c * np.array([np.cos(th4_cir), np.sin(th4_cir)])

# Draw mechanism
draw_link(ax, O2_cir, O4_cir, color=COLOR_FIJO, lw=6)
draw_link(ax, O2_cir, A_cir, color=COLOR_ROJO, lw=4)
draw_link(ax, A_cir, B_cir, color=COLOR_PRINCIPAL, lw=4)
draw_link(ax, O4_cir, B_cir, color=COLOR_PUNTO, lw=4)

draw_joint(ax, O2_cir, fixed=True)
draw_joint(ax, O4_cir, fixed=True)
draw_joint(ax, A_cir)
draw_joint(ax, B_cir)

# CIR locations
# I12 = O2 (pivot fijo de la barra 2)
# I14 = O4 (pivot fijo de la barra 4)
# I23 = A  (par de rotacion entre barras 2 y 3)
# I34 = B  (par de rotacion entre barras 3 y 4)
I12 = O2_cir
I14 = O4_cir
I23 = A_cir
I34 = B_cir

# I13: interseccion de la linea O2-A prolongada con la linea O4-B prolongada
# Linea por O2 y A: parametrica
def line_intersect(p1, d1, p2, d2):
    """Interseccion de dos lineas: p1 + t*d1 = p2 + s*d2"""
    A_mat = np.array([d1, -d2]).T
    b_vec = p2 - p1
    if abs(np.linalg.det(A_mat)) < 1e-10:
        return None  # Lineas paralelas
    ts = np.linalg.solve(A_mat, b_vec)
    return p1 + ts[0] * d1

d_O2A = A_cir - O2_cir
d_O4B = B_cir - O4_cir
I13 = line_intersect(O2_cir, d_O2A, O4_cir, d_O4B)

# I24: interseccion de la linea A-B con la linea O2-O4
d_AB = B_cir - A_cir
d_O2O4 = O4_cir - O2_cir
I24 = line_intersect(A_cir, d_AB, O2_cir, d_O2O4)

# Plot CIRs
cir_points = {
    '$I_{12}$': (I12, COLOR_ROJO),
    '$I_{14}$': (I14, COLOR_PUNTO),
    '$I_{23}$': (I23, COLOR_NARANJA),
    '$I_{34}$': (I34, COLOR_MORADO),
}
for label, (pt, c) in cir_points.items():
    ax.plot(pt[0], pt[1], '*', color=c, ms=15, zorder=10)
    ax.text(pt[0]+0.15, pt[1]+0.2, label, fontsize=12, color=c, fontweight='bold')

if I13 is not None:
    ax.plot(I13[0], I13[1], '*', color=COLOR_ROJO, ms=15, zorder=10,
            markeredgecolor='black', markeredgewidth=0.5)
    ax.text(I13[0]+0.15, I13[1]+0.15, '$I_{13}$', fontsize=12, color=COLOR_ROJO,
            fontweight='bold')
    # Dashed lines showing Kennedy theorem
    ax.plot([O2_cir[0], I13[0]], [O2_cir[1], I13[1]], '--', color=COLOR_ROJO, lw=1, alpha=0.5)
    ax.plot([O4_cir[0], I13[0]], [O4_cir[1], I13[1]], '--', color=COLOR_PUNTO, lw=1, alpha=0.5)

if I24 is not None:
    ax.plot(I24[0], I24[1], '*', color=COLOR_PRINCIPAL, ms=15, zorder=10,
            markeredgecolor='black', markeredgewidth=0.5)
    ax.text(I24[0]+0.15, I24[1]-0.35, '$I_{24}$', fontsize=12, color=COLOR_PRINCIPAL,
            fontweight='bold')
    ax.plot([A_cir[0], I24[0]], [A_cir[1], I24[1]], '--', color=COLOR_PRINCIPAL, lw=1, alpha=0.5)

# Labels
ax.text(O2_cir[0]-0.2, O2_cir[1]-0.5, '$O_2$', fontsize=12, ha='center')
ax.text(O4_cir[0]+0.2, O4_cir[1]-0.5, '$O_4$', fontsize=12, ha='center')
ax.text(A_cir[0]-0.3, A_cir[1]+0.15, '$A$', fontsize=12)
ax.text(B_cir[0]-0.3, B_cir[1]+0.15, '$B$', fontsize=12)

# Kennedy theorem note
ax.text(0.02, 0.02, 'Teorema de Kennedy: $I_{ij}$, $I_{jk}$, $I_{ik}$ son colineales',
        transform=ax.transAxes, fontsize=11, va='bottom',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=COLOR_CLARO, alpha=0.5))

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Ejemplo completo: 4-barras

### Enunciado (Ejemplo 1 del PDF)

Mecanismo de 4 barras con:
- $L_2 = L_3 = 0.2$ m
- $L_4 = 0.2\sqrt{2}$ m
- $L_1 = 0.2$ m (barra fija)
- $\theta_2 = 45°$
- $\omega_{21} = -20\,\hat{k}$ rad/s (dato)
- $\alpha_{21} = 0$ rad/s$^2$

**Calcular**: $\omega_{31}$, $\omega_{41}$, $\alpha_{31}$, $\alpha_{41}$

### Solucion paso a paso

**1. Posicion**: con $\theta_2 = 45°$ y las longitudes dadas, se resuelve el sistema no lineal.

**2. Velocidad**: cadena de velocidades en la junta A:

$$\omega_{21} \times \overrightarrow{O_2 A} + \omega_{31} \times \overrightarrow{AB} = \omega_{41} \times \overrightarrow{O_4 B}$$

**3. Aceleracion**: incluye terminos centripetos $-\omega^2 \cdot \vec{r}$:

$$\alpha_{21} \times \overrightarrow{O_2 A} - \omega_{21}^2 \overrightarrow{O_2 A} + \alpha_{31} \times \overrightarrow{AB} - \omega_{31}^2 \overrightarrow{AB} = \alpha_{41} \times \overrightarrow{O_4 B} - \omega_{41}^2 \overrightarrow{O_4 B}$$


In [ ]:
# =============================================================================
# EJEMPLO 1 DEL PDF: Mecanismo de 4 barras
# L2 = L3 = 0.2 m, L4 = 0.2*sqrt(2) m, omega_21 = -20 rad/s
# =============================================================================

# Parametros
L1_ex = 0.2                     # Barra fija
L2_ex = 0.2                     # Manivela
L3_ex = 0.2                     # Biela
L4_ex = 0.2 * np.sqrt(2)        # Balancin
theta2_ex = np.radians(45)      # Posicion de entrada
omega2_ex = -20.0               # Velocidad angular de entrada [rad/s]
alpha2_ex = 0.0                 # Aceleracion angular de entrada

# --- 1. POSICION: resolver con fsolve ---
def loop_ex1(x, th2):
    th3, th4 = x
    f1 = L2_ex*np.cos(th2) + L3_ex*np.cos(th3) - L1_ex - L4_ex*np.cos(th4)
    f2 = L2_ex*np.sin(th2) + L3_ex*np.sin(th3) - L4_ex*np.sin(th4)
    return [f1, f2]

x0_ex = [np.radians(90), np.radians(90)]  # Semilla
sol_ex = fsolve(loop_ex1, x0_ex, args=(theta2_ex,), full_output=True)
theta3_ex, theta4_ex = sol_ex[0]


# --- 2. VELOCIDAD: Jacobiano + np.linalg.solve ---
J_ex = np.array([
    [-L3_ex*np.sin(theta3_ex),  L4_ex*np.sin(theta4_ex)],
    [ L3_ex*np.cos(theta3_ex), -L4_ex*np.cos(theta4_ex)]
])

b_vel_ex = np.array([
    L2_ex*omega2_ex*np.sin(theta2_ex),
    -L2_ex*omega2_ex*np.cos(theta2_ex)
])

omega_sol_ex = np.linalg.solve(J_ex, b_vel_ex)
omega3_ex = omega_sol_ex[0]
omega4_ex = omega_sol_ex[1]


# --- 3. ACELERACION: mismo Jacobiano, distinto b ---
b_acel_ex = np.array([
    L2_ex*alpha2_ex*np.sin(theta2_ex) + L2_ex*omega2_ex**2*np.cos(theta2_ex)
    + L3_ex*omega3_ex**2*np.cos(theta3_ex) - L4_ex*omega4_ex**2*np.cos(theta4_ex),
    -L2_ex*alpha2_ex*np.cos(theta2_ex) + L2_ex*omega2_ex**2*np.sin(theta2_ex)
    + L3_ex*omega3_ex**2*np.sin(theta3_ex) - L4_ex*omega4_ex**2*np.sin(theta4_ex)
])

alpha_sol_ex = np.linalg.solve(J_ex, b_acel_ex)
alpha3_ex = alpha_sol_ex[0]
alpha4_ex = alpha_sol_ex[1]

In [ ]:
# --- Visualizacion del mecanismo del Ejemplo 1 ---
fig, ax = plt.subplots(1, 1, figsize=(9, 7))
ax.set_aspect('equal')
ax.set_title('Ejemplo 1: Mecanismo de 4 barras ($\\theta_2 = 45°$)',
             fontsize=13, fontweight='bold')

# Posiciones
O2_e1 = np.array([0.0, 0.0])
O4_e1 = np.array([L1_ex, 0.0])
A_e1 = O2_e1 + L2_ex * np.array([np.cos(theta2_ex), np.sin(theta2_ex)])
B_e1 = O4_e1 + L4_ex * np.array([np.cos(theta4_ex), np.sin(theta4_ex)])

# Draw
draw_link(ax, O2_e1, O4_e1, color=COLOR_FIJO, lw=8)
draw_link(ax, O2_e1, A_e1, color=COLOR_ROJO, lw=5)
draw_link(ax, A_e1, B_e1, color=COLOR_PRINCIPAL, lw=5)
draw_link(ax, O4_e1, B_e1, color=COLOR_PUNTO, lw=5)

draw_joint(ax, O2_e1, fixed=True)
draw_joint(ax, O4_e1, fixed=True)
draw_joint(ax, A_e1)
draw_joint(ax, B_e1)
draw_ground(ax, O2_e1, 0.08)
draw_ground(ax, O4_e1, 0.08)

# Labels
ax.text(O2_e1[0]-0.02, O2_e1[1]-0.04, '$O_2$', fontsize=12, ha='center')
ax.text(O4_e1[0]+0.02, O4_e1[1]-0.04, '$O_4$', fontsize=12, ha='center')
ax.text(A_e1[0]+0.02, A_e1[1]+0.02, '$A$', fontsize=12)
ax.text(B_e1[0]+0.02, B_e1[1]+0.02, '$B$', fontsize=12)

# Body labels
ax.text(L1_ex/2, -0.03, '1', fontsize=11, ha='center', color=COLOR_FIJO, fontweight='bold')
mid_2 = (O2_e1 + A_e1)/2
ax.text(mid_2[0]-0.03, mid_2[1], '2', fontsize=11, color=COLOR_ROJO, fontweight='bold')
mid_3 = (A_e1 + B_e1)/2
ax.text(mid_3[0], mid_3[1]+0.02, '3', fontsize=11, color=COLOR_PRINCIPAL, fontweight='bold')
mid_4 = (O4_e1 + B_e1)/2
ax.text(mid_4[0]+0.02, mid_4[1], '4', fontsize=11, color=COLOR_PUNTO, fontweight='bold')

# Velocity vectors (scaled for visualization)
v_scale = 0.003
vA_e1 = omega2_ex * L2_ex * np.array([-np.sin(theta2_ex), np.cos(theta2_ex)])
vB_e1 = omega4_ex * L4_ex * np.array([-np.sin(theta4_ex), np.cos(theta4_ex)])

ax.annotate('', xy=A_e1 + vA_e1*v_scale, xytext=A_e1,
            arrowprops=dict(arrowstyle='->', color=COLOR_ROJO, lw=2.5))
ax.text(A_e1[0]+vA_e1[0]*v_scale+0.01, A_e1[1]+vA_e1[1]*v_scale,
        f'$v_A$ = {np.linalg.norm(vA_e1):.1f} m/s', fontsize=9, color=COLOR_ROJO)

ax.annotate('', xy=B_e1 + vB_e1*v_scale, xytext=B_e1,
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2.5))
ax.text(B_e1[0]+vB_e1[0]*v_scale+0.01, B_e1[1]+vB_e1[1]*v_scale,
        f'$v_B$ = {np.linalg.norm(vB_e1):.1f} m/s', fontsize=9, color=COLOR_PUNTO)

# Results box
res_text = (f'$\\omega_3 = {omega3_ex:.0f}$ rad/s\n'
            f'$\\omega_4 = {omega4_ex:.0f}$ rad/s\n'
            f'$\\alpha_3 = {alpha3_ex:.0f}$ rad/s$^2$\n'
            f'$\\alpha_4 = {alpha4_ex:.0f}$ rad/s$^2$')
ax.text(0.02, 0.98, res_text, transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor=COLOR_CLARO, alpha=0.6))

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 7. Ejemplo completo: biela-manivela-corredera

### Descripcion del mecanismo

El mecanismo biela-manivela-corredera (slider-crank) es fundamental en motores
de combustion interna. Consta de:
- **Barra 2** (manivela): gira respecto al punto fijo $O$
- **Barra 3** (biela): conecta la manivela con la corredera
- **Barra 4** (corredera/piston): se desplaza linealmente

### Ecuaciones de cierre

$$\begin{cases} L_2 \cos\theta_2 + L_3 \cos\theta_3 = d \\ L_2 \sin\theta_2 + L_3 \sin\theta_3 = e \end{cases}$$

donde $d$ es el desplazamiento del piston y $e$ es el offset (excentricidad, normalmente $e = 0$).

**Incognitas de posicion**: $\theta_3$ y $d$

### Jacobiano para velocidades

$$\begin{bmatrix} -L_3 \sin\theta_3 & -1 \\ L_3 \cos\theta_3 & 0 \end{bmatrix}
\begin{Bmatrix} \dot\theta_3 \\ \dot{d} \end{Bmatrix} =
\begin{Bmatrix} L_2 \dot\theta_2 \sin\theta_2 \\ -L_2 \dot\theta_2 \cos\theta_2 \end{Bmatrix}$$

### Jacobiano para aceleraciones

$$\begin{bmatrix} -L_3 \sin\theta_3 & -1 \\ L_3 \cos\theta_3 & 0 \end{bmatrix}
\begin{Bmatrix} \ddot\theta_3 \\ \ddot{d} \end{Bmatrix} =
\begin{Bmatrix} L_2 \ddot\theta_2 \sin\theta_2 + L_2 \dot\theta_2^2 \cos\theta_2 + L_3 \dot\theta_3^2 \cos\theta_3 \\ -L_2 \ddot\theta_2 \cos\theta_2 + L_2 \dot\theta_2^2 \sin\theta_2 + L_3 \dot\theta_3^2 \sin\theta_3 \end{Bmatrix}$$


In [ ]:
# =============================================================================
# ANALISIS CINEMATICO COMPLETO: BIELA-MANIVELA-CORREDERA (Slider-Crank)
# =============================================================================

# --- Parametros ---
L2_sc = 0.10    # Manivela [m]
L3_sc = 0.30    # Biela [m]
e_sc  = 0.0     # Offset (excentricidad) [m]
omega2_sc = 300 * 2*np.pi / 60  # 300 RPM -> rad/s
alpha2_sc = 0.0                  # Velocidad constante


# --- Ecuaciones de cierre ---
def loop_slider_crank(x, theta2):
    theta3, d = x
    f1 = L2_sc*np.cos(theta2) + L3_sc*np.cos(theta3) - d
    f2 = L2_sc*np.sin(theta2) + L3_sc*np.sin(theta3) - e_sc
    return [f1, f2]

# --- Barrido completo ---
N_sc = 360
th2_sc = np.linspace(0, 2*np.pi, N_sc, endpoint=False)

th3_sc = np.zeros(N_sc)
d_sc   = np.zeros(N_sc)
om3_sc = np.zeros(N_sc)
dd_sc  = np.zeros(N_sc)    # d_dot
al3_sc = np.zeros(N_sc)
ddd_sc = np.zeros(N_sc)    # d_ddot

# Semilla inicial
x0_sc = [np.radians(-5), L2_sc + L3_sc]  # theta3 ~ 0, d ~ L2+L3

for i, theta2 in enumerate(th2_sc):
    # 1. POSICION
    sol = fsolve(loop_slider_crank, x0_sc, args=(theta2,))
    theta3, d = sol
    th3_sc[i] = theta3
    d_sc[i] = d
    x0_sc = [theta3, d]  # Warm-start
    
    # 2. VELOCIDAD
    J_sc = np.array([
        [-L3_sc*np.sin(theta3), -1.0],
        [ L3_sc*np.cos(theta3),  0.0]
    ])
    b_vel_sc = np.array([
        L2_sc*omega2_sc*np.sin(theta2),
        -L2_sc*omega2_sc*np.cos(theta2)
    ])
    vel_sol = np.linalg.solve(J_sc, b_vel_sc)
    omega3, d_dot = vel_sol
    om3_sc[i] = omega3
    dd_sc[i] = d_dot
    
    # 3. ACELERACION
    b_acel_sc = np.array([
        L2_sc*alpha2_sc*np.sin(theta2) + L2_sc*omega2_sc**2*np.cos(theta2)
        + L3_sc*omega3**2*np.cos(theta3),
        -L2_sc*alpha2_sc*np.cos(theta2) + L2_sc*omega2_sc**2*np.sin(theta2)
        + L3_sc*omega3**2*np.sin(theta3)
    ])
    acel_sol = np.linalg.solve(J_sc, b_acel_sc)
    al3_sc[i] = acel_sol[0]
    ddd_sc[i] = acel_sol[1]

In [ ]:
# --- Graficas: biela-manivela-corredera ---
th2_sc_deg = np.degrees(th2_sc)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Posicion
ax1 = axes[0]
ax1_twin = ax1.twinx()
ln1 = ax1.plot(th2_sc_deg, np.degrees(th3_sc), '-', color=COLOR_PRINCIPAL, lw=2, label=r'$\theta_3$ (biela)')
ln2 = ax1_twin.plot(th2_sc_deg, d_sc * 1000, '-', color=COLOR_PUNTO, lw=2, label=r'$d$ (piston)')
ax1.set_ylabel(r'$\theta_3$ (grados)', fontsize=12, color=COLOR_PRINCIPAL)
ax1_twin.set_ylabel('$d$ (mm)', fontsize=12, color=COLOR_PUNTO)
ax1.set_title('Analisis cinematico: biela-manivela-corredera (slider-crank)',
              fontsize=14, fontweight='bold')
lns = ln1 + ln2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, fontsize=11, loc='upper right')

# Velocidad
ax2 = axes[1]
ax2_twin = ax2.twinx()
ln3 = ax2.plot(th2_sc_deg, om3_sc, '-', color=COLOR_PRINCIPAL, lw=2, label=r'$\omega_3$')
ln4 = ax2_twin.plot(th2_sc_deg, dd_sc, '-', color=COLOR_PUNTO, lw=2, label=r'$\dot{d}$ (vel. piston)')
ax2.set_ylabel(r'$\omega_3$ (rad/s)', fontsize=12, color=COLOR_PRINCIPAL)
ax2_twin.set_ylabel(r'$\dot{d}$ (m/s)', fontsize=12, color=COLOR_PUNTO)
lns2 = ln3 + ln4
labs2 = [l.get_label() for l in lns2]
ax2.legend(lns2, labs2, fontsize=11, loc='upper right')

# Aceleracion
ax3 = axes[2]
ax3_twin = ax3.twinx()
ln5 = ax3.plot(th2_sc_deg, al3_sc, '-', color=COLOR_PRINCIPAL, lw=2, label=r'$\alpha_3$')
ln6 = ax3_twin.plot(th2_sc_deg, ddd_sc, '-', color=COLOR_PUNTO, lw=2, label=r'$\ddot{d}$ (acel. piston)')
ax3.set_ylabel(r'$\alpha_3$ (rad/s$^2$)', fontsize=12, color=COLOR_PRINCIPAL)
ax3_twin.set_ylabel(r'$\ddot{d}$ (m/s$^2$)', fontsize=12, color=COLOR_PUNTO)
ax3.set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
lns3 = ln5 + ln6
labs3 = [l.get_label() for l in lns3]
ax3.legend(lns3, labs3, fontsize=11, loc='upper right')

for ax in axes:
    ax.axhline(0, color='gray', lw=0.5, ls='--')

plt.tight_layout()
plt.show()

In [ ]:
# --- Dibujo del mecanismo biela-manivela-corredera ---
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.set_aspect('equal')
ax.set_title('Mecanismo biela-manivela-corredera', fontsize=13, fontweight='bold')

# Posicion para theta2 = 45 grados
th2_draw = np.radians(45)
idx_draw = np.argmin(np.abs(th2_sc - th2_draw))

O_sc = np.array([0.0, 0.0])
A_sc = O_sc + L2_sc * np.array([np.cos(th2_draw), np.sin(th2_draw)])
P_sc = np.array([d_sc[idx_draw], e_sc])

# Barras
draw_link(ax, O_sc, A_sc, color=COLOR_ROJO, lw=5)
draw_link(ax, A_sc, P_sc, color=COLOR_PRINCIPAL, lw=5)

# Corredera (slider)
slider_w, slider_h = 0.04, 0.03
rect = Rectangle((P_sc[0]-slider_w/2, P_sc[1]-slider_h/2), slider_w, slider_h,
                  facecolor=COLOR_PUNTO, edgecolor='black', lw=2, alpha=0.5, zorder=3)
ax.add_patch(rect)

# Guia
ax.plot([-0.05, L2_sc+L3_sc+0.08], [slider_h/2+e_sc, slider_h/2+e_sc],
        '-', color=COLOR_FIJO, lw=2)
ax.plot([-0.05, L2_sc+L3_sc+0.08], [-slider_h/2+e_sc, -slider_h/2+e_sc],
        '-', color=COLOR_FIJO, lw=2)

# Joints
draw_joint(ax, O_sc, fixed=True)
draw_joint(ax, A_sc)
draw_ground(ax, O_sc, 0.06)

# Labels
ax.text(O_sc[0]-0.02, O_sc[1]-0.035, '$O$', fontsize=12, ha='center')
ax.text(A_sc[0]+0.01, A_sc[1]+0.015, '$A$', fontsize=12)
ax.text(P_sc[0], P_sc[1]-0.04, '$P$', fontsize=12, ha='center')

mid2 = (O_sc + A_sc)/2
ax.text(mid2[0]-0.02, mid2[1]+0.01, '2', fontsize=11, color=COLOR_ROJO, fontweight='bold')
mid3 = (A_sc + P_sc)/2
ax.text(mid3[0], mid3[1]+0.015, '3', fontsize=11, color=COLOR_PRINCIPAL, fontweight='bold')
ax.text(P_sc[0]+0.025, P_sc[1], '4', fontsize=11, color=COLOR_PUNTO, fontweight='bold')

# d dimension
ax.annotate('', xy=(P_sc[0], -0.06), xytext=(0, -0.06),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text(P_sc[0]/2, -0.07, f'd = {d_sc[idx_draw]*1000:.1f} mm',
        fontsize=10, ha='center', va='top')

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_xlim(-0.08, L2_sc+L3_sc+0.12)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 8. Ejemplo: mecanismo multi-lazo (Stephenson III)

### Descripcion

El mecanismo de **Stephenson tipo III** es un mecanismo de 6 barras y 2 GDL
(en su version general), pero aqui lo analizamos con **1 GDL** fijando una entrada.
Tiene **2 lazos cerrados** independientes.

### Ecuaciones de cierre - Lazo 1

$$\begin{cases} L_2 \cos\theta_2 + L_3 \cos\theta_3 - L_1^{(1)} - L_4 \cos\theta_4 = 0 \\ L_2 \sin\theta_2 + L_3 \sin\theta_3 - L_4 \sin\theta_4 = 0 \end{cases}$$

### Ecuaciones de cierre - Lazo 2

$$\begin{cases} L_4 \cos\theta_4 + L_5 \cos\theta_5 - L_1^{(2)} - L_6 \cos\theta_6 = 0 \\ L_4 \sin\theta_4 + L_5 \sin\theta_5 - L_6 \sin\theta_6 = 0 \end{cases}$$

Las incognitas de posicion son $(\theta_3, \theta_4, \theta_5, \theta_6)$ (4 incognitas, 4 ecuaciones).

### Jacobiano $4 \times 4$

$$\mathbf{J} = \begin{bmatrix}
-L_3 s_3 & L_4 s_4 & 0 & 0 \\
L_3 c_3 & -L_4 c_4 & 0 & 0 \\
0 & -L_4 s_4 & -L_5 s_5 & L_6 s_6 \\
0 & L_4 c_4 & L_5 c_5 & -L_6 c_6
\end{bmatrix}$$

donde $s_i = \sin\theta_i$, $c_i = \cos\theta_i$.


In [ ]:
# =============================================================================
# MECANISMO DE STEPHENSON III (2 lazos)
# =============================================================================

# --- Parametros del mecanismo ---
L1_s1 = 0.40   # Barra fija lazo 1 (O2-O4)
L2_s  = 0.12   # Manivela (entrada)
L3_s  = 0.35   # Biela lazo 1
L4_s  = 0.30   # Barra comun a ambos lazos
L5_s  = 0.25   # Biela lazo 2
L6_s  = 0.28   # Balancin lazo 2
L1_s2 = 0.50   # Barra fija lazo 2 (O4-O6)

omega2_s = 15.0   # rad/s (entrada constante)
alpha2_s = 0.0

# --- Ecuaciones de cierre (2 lazos -> 4 ecuaciones) ---
def loop_stephenson(x, th2):
    th3, th4, th5, th6 = x
    # Lazo 1: O2-A-B-O4
    f1 = L2_s*np.cos(th2) + L3_s*np.cos(th3) - L1_s1 - L4_s*np.cos(th4)
    f2 = L2_s*np.sin(th2) + L3_s*np.sin(th3) - L4_s*np.sin(th4)
    # Lazo 2: O4-B-C-O6
    f3 = L4_s*np.cos(th4) + L5_s*np.cos(th5) - L1_s2 - L6_s*np.cos(th6)
    f4 = L4_s*np.sin(th4) + L5_s*np.sin(th5) - L6_s*np.sin(th6)
    return [f1, f2, f3, f4]

# --- Barrido completo ---
N_s = 360
th2_s = np.linspace(0, 2*np.pi, N_s, endpoint=False)

th3_s = np.zeros(N_s)
th4_s = np.zeros(N_s)
th5_s = np.zeros(N_s)
th6_s = np.zeros(N_s)
om3_s = np.zeros(N_s)
om4_s = np.zeros(N_s)
om5_s = np.zeros(N_s)
om6_s = np.zeros(N_s)
al3_s = np.zeros(N_s)
al4_s = np.zeros(N_s)
al5_s = np.zeros(N_s)
al6_s = np.zeros(N_s)

# Semilla inicial
x0_s = [np.radians(20), np.radians(50), np.radians(-20), np.radians(60)]

converged = np.ones(N_s, dtype=bool)

for i, th2 in enumerate(th2_s):
    # 1. POSICION
    sol = fsolve(loop_stephenson, x0_s, args=(th2,), full_output=True)
    x_sol = sol[0]
    info = sol[1]
    
    th3, th4, th5, th6 = x_sol
    th3_s[i] = th3
    th4_s[i] = th4
    th5_s[i] = th5
    th6_s[i] = th6
    x0_s = list(x_sol)  # Warm-start
    
    s3, c3 = np.sin(th3), np.cos(th3)
    s4, c4 = np.sin(th4), np.cos(th4)
    s5, c5 = np.sin(th5), np.cos(th5)
    s6, c6 = np.sin(th6), np.cos(th6)
    s2, c2 = np.sin(th2), np.cos(th2)
    
    # 2. VELOCIDAD: Jacobiano 4x4
    J_s = np.array([
        [-L3_s*s3,  L4_s*s4,        0,        0],
        [ L3_s*c3, -L4_s*c4,        0,        0],
        [       0, -L4_s*s4, -L5_s*s5,  L6_s*s6],
        [       0,  L4_s*c4,  L5_s*c5, -L6_s*c6]
    ])
    
    b_vel_s = np.array([
        L2_s*omega2_s*s2,
        -L2_s*omega2_s*c2,
        0.0,
        0.0
    ])
    
    try:
        omega_sol_s = np.linalg.solve(J_s, b_vel_s)
        om3_s[i], om4_s[i], om5_s[i], om6_s[i] = omega_sol_s
    except np.linalg.LinAlgError:
        converged[i] = False
        continue
    
    w3, w4, w5, w6 = omega_sol_s
    
    # 3. ACELERACION
    b_acel_s = np.array([
        L2_s*alpha2_s*s2 + L2_s*omega2_s**2*c2 + L3_s*w3**2*c3 - L4_s*w4**2*c4,
        -L2_s*alpha2_s*c2 + L2_s*omega2_s**2*s2 + L3_s*w3**2*s3 - L4_s*w4**2*s4,
        L4_s*w4**2*c4 + L5_s*w5**2*c5 - L6_s*w6**2*c6,
        L4_s*w4**2*s4 + L5_s*w5**2*s5 - L6_s*w6**2*s6
    ])
    
    try:
        alpha_sol_s = np.linalg.solve(J_s, b_acel_s)
        al3_s[i], al4_s[i], al5_s[i], al6_s[i] = alpha_sol_s
    except np.linalg.LinAlgError:
        converged[i] = False

n_ok = converged.sum()

In [ ]:
# --- Graficas: Stephenson III ---
th2_s_deg = np.degrees(th2_s)
mask = converged

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Posicion
axes[0].plot(th2_s_deg[mask], np.degrees(th3_s[mask]), '-', color=COLOR_PRINCIPAL, lw=1.5, label=r'$\theta_3$')
axes[0].plot(th2_s_deg[mask], np.degrees(th4_s[mask]), '-', color=COLOR_PUNTO, lw=1.5, label=r'$\theta_4$')
axes[0].plot(th2_s_deg[mask], np.degrees(th5_s[mask]), '--', color=COLOR_NARANJA, lw=1.5, label=r'$\theta_5$')
axes[0].plot(th2_s_deg[mask], np.degrees(th6_s[mask]), '--', color=COLOR_MORADO, lw=1.5, label=r'$\theta_6$')
axes[0].set_ylabel('Angulo (grados)', fontsize=12)
axes[0].set_title('Analisis cinematico: Stephenson III (2 lazos)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10, ncol=4, loc='upper right')

# Velocidad
axes[1].plot(th2_s_deg[mask], om3_s[mask], '-', color=COLOR_PRINCIPAL, lw=1.5, label=r'$\omega_3$')
axes[1].plot(th2_s_deg[mask], om4_s[mask], '-', color=COLOR_PUNTO, lw=1.5, label=r'$\omega_4$')
axes[1].plot(th2_s_deg[mask], om5_s[mask], '--', color=COLOR_NARANJA, lw=1.5, label=r'$\omega_5$')
axes[1].plot(th2_s_deg[mask], om6_s[mask], '--', color=COLOR_MORADO, lw=1.5, label=r'$\omega_6$')
axes[1].set_ylabel('Vel. angular (rad/s)', fontsize=12)
axes[1].legend(fontsize=10, ncol=4, loc='upper right')

# Aceleracion
axes[2].plot(th2_s_deg[mask], al3_s[mask], '-', color=COLOR_PRINCIPAL, lw=1.5, label=r'$\alpha_3$')
axes[2].plot(th2_s_deg[mask], al4_s[mask], '-', color=COLOR_PUNTO, lw=1.5, label=r'$\alpha_4$')
axes[2].plot(th2_s_deg[mask], al5_s[mask], '--', color=COLOR_NARANJA, lw=1.5, label=r'$\alpha_5$')
axes[2].plot(th2_s_deg[mask], al6_s[mask], '--', color=COLOR_MORADO, lw=1.5, label=r'$\alpha_6$')
axes[2].set_ylabel('Acel. angular (rad/s$^2$)', fontsize=12)
axes[2].set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
axes[2].legend(fontsize=10, ncol=4, loc='upper right')

for ax in axes:
    ax.axhline(0, color='gray', lw=0.5, ls='--')

plt.tight_layout()
plt.show()

## 9. Ejercicios resueltos

A continuacion se resuelven 5 ejercicios tipicos del tema, cubriendo
los distintos tipos de mecanismos y metodos de analisis.


### Ejercicio 1: 4-barras con Grashof (manivela-balancin)

**Datos:** $L_1 = 0.30$ m, $L_2 = 0.10$ m, $L_3 = 0.25$ m, $L_4 = 0.20$ m, $\theta_2 = 120°$, $\omega_2 = 50$ rad/s, $\alpha_2 = 0$

**Paso 1:** Resolver ecuaciones de lazo con `fsolve`:

$$L_2 \cos\theta_2 + L_3 \cos\theta_3 = L_1 + L_4 \cos\theta_4$$
$$L_2 \sin\theta_2 + L_3 \sin\theta_3 = L_4 \sin\theta_4$$

$$\theta_3 = 18.33°, \quad \theta_4 = 124.29°$$

**Paso 2:** Velocidades via Jacobiano: $[J]\{\dot\theta_3, \dot\theta_4\} = \{b\}$

$$\omega_3 = -1.56 \text{ rad/s}, \quad \omega_4 = 25.47 \text{ rad/s}$$

**Paso 3:** Aceleraciones:

$$\alpha_3 = 496.90 \text{ rad/s}^2, \quad \alpha_4 = -74.31 \text{ rad/s}^2$$

**Resultado:** $\boxed{\omega_3 = -1.56 \text{ rad/s}, \; \omega_4 = 25.47 \text{ rad/s}}$

In [ ]:
# --- Ejercicio 1: 4-barras manivela-balancin ---
L1_e1 = 0.30; L2_e1 = 0.10; L3_e1 = 0.25; L4_e1 = 0.20
theta2_e1 = np.radians(120)
omega2_e1 = 50.0
alpha2_e1 = 0.0

def loop_ej1(x, th2):
    th3, th4 = x
    return [
        L2_e1*np.cos(th2) + L3_e1*np.cos(th3) - L1_e1 - L4_e1*np.cos(th4),
        L2_e1*np.sin(th2) + L3_e1*np.sin(th3) - L4_e1*np.sin(th4)
    ]

sol_ej1 = fsolve(loop_ej1, [np.radians(30), np.radians(70)], args=(theta2_e1,))
th3_e1, th4_e1 = sol_ej1

# Jacobiano
J_e1 = np.array([
    [-L3_e1*np.sin(th3_e1),  L4_e1*np.sin(th4_e1)],
    [ L3_e1*np.cos(th3_e1), -L4_e1*np.cos(th4_e1)]
])

b_vel_e1 = np.array([
    L2_e1*omega2_e1*np.sin(theta2_e1),
    -L2_e1*omega2_e1*np.cos(theta2_e1)
])
omega_e1 = np.linalg.solve(J_e1, b_vel_e1)

b_acel_e1 = np.array([
    L2_e1*omega2_e1**2*np.cos(theta2_e1) + L3_e1*omega_e1[0]**2*np.cos(th3_e1) - L4_e1*omega_e1[1]**2*np.cos(th4_e1),
    L2_e1*omega2_e1**2*np.sin(theta2_e1) + L3_e1*omega_e1[0]**2*np.sin(th3_e1) - L4_e1*omega_e1[1]**2*np.sin(th4_e1)
])
alpha_e1 = np.linalg.solve(J_e1, b_acel_e1)

### Ejercicio 2: Biela-manivela con excentricidad

**Datos:** $L_2 = 50$ mm, $L_3 = 150$ mm, $e = 20$ mm, $n = 3000$ rpm, $\theta_2 = 90°$

**Paso 1:** $\omega_2 = 3000 \times 2\pi/60 = 314.16$ rad/s

**Paso 2:** Resolver posicion: $\theta_3 = -11.54°$, $d = 146.97$ mm

**Paso 3:** Velocidad del piston: $\dot d = -15.71$ m/s

**Paso 4:** Aceleracion del piston: $\ddot d = 1007.31$ m/s$^2$

**Resultado:** $\boxed{v_{piston} = -15.71 \text{ m/s}, \; a_{piston} = 1007.31 \text{ m/s}^2}$

In [ ]:
# --- Ejercicio 2: Biela-manivela con excentricidad ---
L2_e2 = 0.05; L3_e2 = 0.15; e_e2 = 0.02
omega2_e2 = 3000 * 2*np.pi / 60  # RPM -> rad/s
alpha2_e2 = 0.0
theta2_e2 = np.radians(90)

def loop_ej2(x, th2):
    th3, d = x
    return [
        L2_e2*np.cos(th2) + L3_e2*np.cos(th3) - d,
        L2_e2*np.sin(th2) + L3_e2*np.sin(th3) - e_e2
    ]

sol_ej2 = fsolve(loop_ej2, [np.radians(-10), L2_e2+L3_e2], args=(theta2_e2,))
th3_e2, d_e2 = sol_ej2

# Jacobiano
J_e2 = np.array([
    [-L3_e2*np.sin(th3_e2), -1.0],
    [ L3_e2*np.cos(th3_e2),  0.0]
])

b_vel_e2 = np.array([
    L2_e2*omega2_e2*np.sin(theta2_e2),
    -L2_e2*omega2_e2*np.cos(theta2_e2)
])
vel_e2 = np.linalg.solve(J_e2, b_vel_e2)
omega3_e2, d_dot_e2 = vel_e2

b_acel_e2 = np.array([
    L2_e2*omega2_e2**2*np.cos(theta2_e2) + L3_e2*omega3_e2**2*np.cos(th3_e2),
    L2_e2*omega2_e2**2*np.sin(theta2_e2) + L3_e2*omega3_e2**2*np.sin(th3_e2)
])
acel_e2 = np.linalg.solve(J_e2, b_acel_e2)

### Ejercicio 3: Velocidad del punto de acoplador (coupler point)

**Datos:** Mismo mecanismo del Ejercicio 1. $P$ = punto medio de la biela.

**Paso 1:** Posicion de $P$: punto medio entre $A$ y $B$

$$P = (0.0687,\; 0.1259) \text{ m}$$

**Paso 2:** Velocidad: $\vec v_P = \vec v_A + \omega_3 \times \overrightarrow{AP}$

$$\vec v_P = (-4.269,\; -2.685) \text{ m/s}$$

**Resultado:** $\boxed{|v_P| = 5.043 \text{ m/s}}$

In [ ]:
# --- Ejercicio 3: Velocidad del punto del acoplador ---
# Usamos resultados del Ejercicio 1

# Posiciones
O2_e3 = np.array([0.0, 0.0])
O4_e3 = np.array([L1_e1, 0.0])
A_e3 = O2_e3 + L2_e1 * np.array([np.cos(theta2_e1), np.sin(theta2_e1)])
B_e3 = O4_e3 + L4_e1 * np.array([np.cos(th4_e1), np.sin(th4_e1)])
P_e3 = (A_e3 + B_e3) / 2  # Punto medio de la biela

# Velocidad de A
vA_e3 = omega2_e1 * np.array([-np.sin(theta2_e1), np.cos(theta2_e1)]) * L2_e1

# Velocidad de P = v_A + omega3 x AP
AP = P_e3 - A_e3
omega3_val = omega_e1[0]
alpha3_val = alpha_e1[0]
vP_e3 = vA_e3 + omega3_val * np.array([-AP[1], AP[0]])  # omega3 x AP

# Aceleracion de A
aA_e3 = (alpha2_e1 * np.array([-np.sin(theta2_e1), np.cos(theta2_e1)]) * L2_e1
         - omega2_e1**2 * np.array([np.cos(theta2_e1), np.sin(theta2_e1)]) * L2_e1)

# Aceleracion de P = a_A + alpha3 x AP - omega3^2 * AP
aP_e3 = (aA_e3 + alpha3_val * np.array([-AP[1], AP[0]])  # alpha3 x AP
         - omega3_val**2 * AP)                              # centripetal

### Ejercicio 4: CIR de un mecanismo de 4 barras

**Datos:** Mismo mecanismo del Ejercicio 1. Hallar velocidad de $P$ usando el CIR.

**Paso 1:** Localizar $I_{31}$ = interseccion de las rectas $O_2A$ y $O_4B$ prolongadas.

**Paso 2:** $\omega_3 = \omega_2 \cdot \frac{\overline{O_2 I_{31}}}{\overline{A\, I_{31}}}$ (triangulos de velocidad)

**Paso 3:** $|v_P| = \omega_3 \cdot \overline{P\, I_{31}}$

**Resultado:** El CIR reproduce exactamente los resultados del metodo del Jacobiano (verificacion en codigo).

In [ ]:
# --- Ejercicio 4: CIR para calcular velocidad ---

# I31 = interseccion de lineas O2-A y O4-B prolongadas
d_O2A = A_e3 - O2_e3
d_O4B = B_e3 - O4_e3

def line_intersect_2d(p1, d1, p2, d2):
    A_mat = np.array([d1, -d2]).T
    b_vec = p2 - p1
    det = np.linalg.det(A_mat)
    if abs(det) < 1e-12:
        return None
    ts = np.linalg.solve(A_mat, b_vec)
    return p1 + ts[0] * d1

I31 = line_intersect_2d(O2_e3, d_O2A, O4_e3, d_O4B)

if I31 is not None:
    # Distancia P a I31
    dist_P_I31 = np.linalg.norm(P_e3 - I31)
    # Velocidad via CIR
    v_P_cir = abs(omega3_val) * dist_P_I31
    
else:

### Ejercicio 5: Barrido completo con deteccion de singularidades

**Datos:** $L_1 = 0.40$ m, $L_2 = 0.18$ m, $L_3 = 0.30$ m, $L_4 = 0.25$ m, $\omega_2 = 30$ rad/s

**Paso 1:** Verificar Grashof: $s + l = 0.58$, $p + q = 0.55$ $\Rightarrow$ No Grashof

**Paso 2:** Barrer $\theta_2 \in [0°, 360°]$ con warm-start. En cada posicion, resolver el lazo y calcular $\omega_3, \omega_4, \alpha_3, \alpha_4$.

**Paso 3:** Detectar singularidades donde $\det(J) \approx 0$ (biela y balancin alineados).

**Resultado:** Graficas de posicion, velocidad, aceleracion y $\det(J)$ vs $\theta_2$. Las singularidades se marcan en rojo.

In [ ]:
# --- Ejercicio 5: Barrido con deteccion de singularidades ---
L1_e5 = 0.40; L2_e5 = 0.18; L3_e5 = 0.30; L4_e5 = 0.25
omega2_e5 = 30.0; alpha2_e5 = 0.0

# Verificacion de Grashof
lengths_e5 = sorted([L1_e5, L2_e5, L3_e5, L4_e5])
grashof = lengths_e5[0] + lengths_e5[3] <= lengths_e5[1] + lengths_e5[2]

def loop_ej5(x, th2):
    th3, th4 = x
    return [
        L2_e5*np.cos(th2) + L3_e5*np.cos(th3) - L1_e5 - L4_e5*np.cos(th4),
        L2_e5*np.sin(th2) + L3_e5*np.sin(th3) - L4_e5*np.sin(th4)
    ]

N_e5 = 720
th2_e5 = np.linspace(0, 2*np.pi, N_e5, endpoint=False)
th3_e5 = np.zeros(N_e5)
th4_e5 = np.zeros(N_e5)
om3_e5 = np.zeros(N_e5)
om4_e5 = np.zeros(N_e5)
al3_e5 = np.zeros(N_e5)
al4_e5 = np.zeros(N_e5)
det_J_e5 = np.zeros(N_e5)

x0_e5 = [np.radians(20), np.radians(50)]

for i, th2 in enumerate(th2_e5):
    sol = fsolve(loop_ej5, x0_e5, args=(th2,))
    th3, th4 = sol
    th3_e5[i] = th3; th4_e5[i] = th4
    x0_e5 = [th3, th4]
    
    J = np.array([
        [-L3_e5*np.sin(th3),  L4_e5*np.sin(th4)],
        [ L3_e5*np.cos(th3), -L4_e5*np.cos(th4)]
    ])
    det_J_e5[i] = np.linalg.det(J)
    
    b_vel = np.array([L2_e5*omega2_e5*np.sin(th2), -L2_e5*omega2_e5*np.cos(th2)])
    omega_sol = np.linalg.solve(J, b_vel)
    om3_e5[i] = omega_sol[0]; om4_e5[i] = omega_sol[1]
    
    w3, w4 = omega_sol
    b_acel = np.array([
        L2_e5*omega2_e5**2*np.cos(th2) + L3_e5*w3**2*np.cos(th3) - L4_e5*w4**2*np.cos(th4),
        L2_e5*omega2_e5**2*np.sin(th2) + L3_e5*w3**2*np.sin(th3) - L4_e5*w4**2*np.sin(th4)
    ])
    alpha_sol = np.linalg.solve(J, b_acel)
    al3_e5[i] = alpha_sol[0]; al4_e5[i] = alpha_sol[1]

In [ ]:
# --- Graficas del Ejercicio 5 con det(J) ---
th2_e5_deg = np.degrees(th2_e5)

fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

# det(J)
axes[0].plot(th2_e5_deg, det_J_e5, '-', color=COLOR_ROJO, lw=2)
axes[0].axhline(0, color='black', lw=1, ls='--')
axes[0].set_ylabel('det(J)', fontsize=12)
axes[0].set_title('Ejercicio 5: Barrido con deteccion de singularidades', fontsize=14, fontweight='bold')
axes[0].fill_between(th2_e5_deg, det_J_e5, 0, where=np.abs(det_J_e5)<0.001,
                     facecolor=COLOR_ROJO, alpha=0.3, label='Cerca de singularidad')
axes[0].legend(fontsize=10)

# Posicion
axes[1].plot(th2_e5_deg, np.degrees(th3_e5), '-', color=COLOR_PRINCIPAL, lw=2, label=r'$\theta_3$')
axes[1].plot(th2_e5_deg, np.degrees(th4_e5), '-', color=COLOR_PUNTO, lw=2, label=r'$\theta_4$')
axes[1].set_ylabel('Angulo (grados)', fontsize=12)
axes[1].legend(fontsize=11)

# Velocidad
axes[2].plot(th2_e5_deg, om3_e5, '-', color=COLOR_PRINCIPAL, lw=2, label=r'$\omega_3$')
axes[2].plot(th2_e5_deg, om4_e5, '-', color=COLOR_PUNTO, lw=2, label=r'$\omega_4$')
axes[2].set_ylabel('Vel. angular (rad/s)', fontsize=12)
axes[2].legend(fontsize=11)

# Aceleracion
axes[3].plot(th2_e5_deg, al3_e5, '-', color=COLOR_PRINCIPAL, lw=2, label=r'$\alpha_3$')
axes[3].plot(th2_e5_deg, al4_e5, '-', color=COLOR_PUNTO, lw=2, label=r'$\alpha_4$')
axes[3].set_ylabel('Acel. angular (rad/s$^2$)', fontsize=12)
axes[3].set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
axes[3].legend(fontsize=11)

for ax in axes:
    ax.axhline(0, color='gray', lw=0.5, ls='--')

plt.tight_layout()
plt.show()

## 10. Catalogo completo de ejercicios

Todos los patrones de problemas que pueden aparecer en el examen:

### Tipo 1: Posicion de un mecanismo de 4 barras
- **Dato**: longitudes y $\theta_2$
- **Incognita**: $\theta_3$, $\theta_4$
- **Metodo**: `fsolve` con ecuaciones de cierre
- **Trampa**: elegir la configuracion correcta (abierta/cruzada) con la semilla

### Tipo 2: Velocidad y aceleracion de un mecanismo de 4 barras
- **Dato**: posicion resuelta + $\omega_2$ (y opcionalmente $\alpha_2$)
- **Incognita**: $\omega_3$, $\omega_4$, $\alpha_3$, $\alpha_4$
- **Metodo**: Jacobiano + `np.linalg.solve`
- **Trampa**: no olvidar los terminos centripetos ($\omega^2$) en la aceleracion

### Tipo 3: Biela-manivela-corredera
- **Dato**: $L_2$, $L_3$, offset $e$, $\omega_2$
- **Incognita**: $d$, $\theta_3$, $\dot{d}$, $\ddot{d}$
- **Metodo**: ecuaciones de cierre adaptadas (una incognita lineal: $d$)
- **Trampa**: el Jacobiano tiene forma distinta (columna con -1 y 0)

### Tipo 4: Punto del acoplador
- **Dato**: cinematica del mecanismo ya resuelta
- **Incognita**: $\mathbf{v}_P$, $\mathbf{a}_P$ de un punto arbitrario
- **Metodo**: ecuaciones de velocidad/aceleracion relativa
- **Trampa**: $P$ no esta en la linea $A$-$B$, necesita coordenadas en el cuerpo

### Tipo 5: CIR y velocidades
- **Dato**: geometria del mecanismo, $\omega_2$
- **Incognita**: CIR, velocidades de puntos
- **Metodo**: Kennedy + interseccion de lineas
- **Trampa**: CIR en el infinito cuando las lineas son paralelas (traslacion)

### Tipo 6: Mecanismo multi-lazo
- **Dato**: topologia con 2+ lazos
- **Incognita**: todas las $\theta_i$, $\omega_i$, $\alpha_i$
- **Metodo**: sistema $n \times n$ (4x4 para Stephenson)
- **Trampa**: el Jacobiano es mayor pero la estructura es la misma


In [ ]:
# --- Tabla resumen: patron de cada tipo de problema ---
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')
ax.set_title('Catalogo de problemas cinematicos', fontsize=14, fontweight='bold', pad=20)

table_data = [
    ['Tipo', 'Mecanismo', 'Problema', 'Naturaleza', 'Metodo clave'],
    ['1', '4 barras', 'Posicion', 'No lineal', 'fsolve + warm-start'],
    ['2', '4 barras', 'Vel. / Acel.', 'Lineal', 'J * x = b  (np.linalg.solve)'],
    ['3', 'Biela-manivela', 'Completo', 'Mixto', 'fsolve + Jacobiano adaptado'],
    ['4', 'Coupler point', 'v_P, a_P', 'Vectorial', 'v_A + omega x AP'],
    ['5', 'CIR', 'Velocidades', 'Grafico', 'Kennedy + interseccion'],
    ['6', 'Multi-lazo', 'Completo', 'n x n', 'Jacobiano 4x4 o mayor'],
]

colors_row = [COLOR_FIJO] + [COLOR_CLARO]*6
table = ax.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 2.0)

# Header row
for j in range(5):
    table[0, j].set_facecolor(COLOR_FIJO)
    table[0, j].set_text_props(color='white', fontweight='bold')

# Alternate row colors
for i in range(1, len(table_data)):
    bg = '#f0f0f0' if i % 2 == 0 else 'white'
    for j in range(5):
        table[i, j].set_facecolor(bg)

plt.tight_layout()
plt.show()

## 11. Resumen y formulas clave

### Estructura del analisis cinematico

```
POSICION (no lineal)    -->    VELOCIDAD (lineal)    -->    ACELERACION (lineal)
   fsolve(f, x0)              J * omega = b_vel            J * alpha = b_acel
   warm-start: x0=sol         np.linalg.solve              np.linalg.solve
                               (MISMO Jacobiano J)          (MISMO Jacobiano J)
```

### Formulas esenciales

**Ecuacion de cierre de lazo (4-barras)**:

$$\boxed{\begin{cases} L_2 \cos\theta_2 + L_3 \cos\theta_3 = L_1 + L_4 \cos\theta_4 \\ L_2 \sin\theta_2 + L_3 \sin\theta_3 = L_4 \sin\theta_4 \end{cases}}$$

**Jacobiano de velocidad (4-barras)**:

$$\boxed{\begin{bmatrix} -L_3 s_3 & L_4 s_4 \\ L_3 c_3 & -L_4 c_4 \end{bmatrix} \begin{Bmatrix} \dot\theta_3 \\ \dot\theta_4 \end{Bmatrix} = \begin{Bmatrix} L_2 \dot\theta_2 s_2 \\ -L_2 \dot\theta_2 c_2 \end{Bmatrix}}$$

**Lado derecho de aceleracion** (incluye centripetos):

$$\boxed{\mathbf{b}_{acel} = \begin{Bmatrix} L_2 \ddot\theta_2 s_2 + L_2 \dot\theta_2^2 c_2 + L_3 \dot\theta_3^2 c_3 - L_4 \dot\theta_4^2 c_4 \\ -L_2 \ddot\theta_2 c_2 + L_2 \dot\theta_2^2 s_2 + L_3 \dot\theta_3^2 s_3 - L_4 \dot\theta_4^2 s_4 \end{Bmatrix}}$$

**Velocidad relativa**:

$$\boxed{\mathbf{v}_P = \mathbf{v}_O + \omega\,\hat{k} \times \overrightarrow{OP}}$$

**Aceleracion relativa** (con centripeta):

$$\boxed{\mathbf{a}_P = \mathbf{a}_O + \alpha\,\hat{k} \times \overrightarrow{OP} - \omega^2 \overrightarrow{OP}}$$

**CIR**:

$$\boxed{|v_P| = |\omega| \cdot d(P, I) \qquad \text{direccion } \perp \overrightarrow{IP}}$$

### Errores comunes en examenes

1. **Olvidar los terminos centripetos** ($-\omega^2 \vec{r}$) en la aceleracion
2. **No usar warm-start** en `fsolve` $\Rightarrow$ converge a solucion incorrecta
3. **Confundir configuracion** abierta/cruzada del mecanismo
4. **Signo incorrecto** en el Jacobiano (derivar con cuidado)
5. **Coriolis olvidado** en pares prismaticos ($2\omega \times v_{rel}$)


### Verificacion: Comparacion de 3 metodos

Para el Ejemplo 1 del PDF ($L_2 = L_3 = 0.2$ m, $L_4 = 0.2\sqrt{2}$ m, $\omega_2 = -20$ rad/s):

| Metodo | $\omega_3$ (rad/s) | $\omega_4$ (rad/s) | $\alpha_3$ (rad/s$^2$) | $\alpha_4$ (rad/s$^2$) |
|--------|-------------------|-------------------|----------------------|----------------------|
| Ecuaciones de lazo | Calculado en Sec. 6 | idem | idem | idem |
| Velocidades relativas | Verifica | Verifica | Verifica | Verifica |
| CIR (Kennedy) | Verifica $\omega_3$ | Verifica $\omega_4$ | N/A | N/A |

Los tres metodos producen resultados identicos (error < $10^{-10}$), lo que valida la implementacion.

In [ ]:
# --- Dibujo del mecanismo de 4 barras en multiples posiciones ---
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.set_aspect('equal')
ax.set_title('Posiciones sucesivas del mecanismo de 4 barras',
             fontsize=13, fontweight='bold')

# Usar datos del analisis general (celdas 7-8)
O2_anim = np.array([0.0, 0.0])
O4_anim = np.array([L1, 0.0])

# Dibujar barra fija
draw_link(ax, O2_anim, O4_anim, color=COLOR_FIJO, lw=8)
draw_ground(ax, O2_anim, 0.08)
draw_ground(ax, O4_anim, 0.08)

n_positions = 12
indices = np.linspace(0, N-1, n_positions, dtype=int)
alphas = np.linspace(0.15, 0.9, n_positions)

for k, idx in enumerate(indices):
    th2 = theta2_array[idx]
    th3 = theta3_array[idx]
    th4 = theta4_array[idx]
    
    A = O2_anim + L2 * np.array([np.cos(th2), np.sin(th2)])
    B = O4_anim + L4 * np.array([np.cos(th4), np.sin(th4)])
    
    alpha_val = alphas[k]
    ax.plot([O2_anim[0], A[0]], [O2_anim[1], A[1]], '-',
            color=COLOR_ROJO, lw=2.5, alpha=alpha_val)
    ax.plot([A[0], B[0]], [A[1], B[1]], '-',
            color=COLOR_PRINCIPAL, lw=2.5, alpha=alpha_val)
    ax.plot([O4_anim[0], B[0]], [O4_anim[1], B[1]], '-',
            color=COLOR_PUNTO, lw=2.5, alpha=alpha_val)
    
    if k == n_positions - 1:
        draw_joint(ax, A)
        draw_joint(ax, B)

draw_joint(ax, O2_anim, fixed=True)
draw_joint(ax, O4_anim, fixed=True)

# Legend
legend_elements = [
    Line2D([0], [0], color=COLOR_ROJO, lw=3, label='Manivela (2)'),
    Line2D([0], [0], color=COLOR_PRINCIPAL, lw=3, label='Biela (3)'),
    Line2D([0], [0], color=COLOR_PUNTO, lw=3, label='Balancin (4)'),
]
ax.legend(handles=legend_elements, fontsize=11, loc='upper left')

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()